# 🧠 NEUROSCAN AI — Parkinson's Disease Detection
## Complete Training Notebook
> **Dataset:** Spiral/Wave Drawing Images | **Task:** Binary Classification (Healthy vs Parkinson)  
> **Models:** Custom CNN · VGG16 · ResNet50 · (Optional) YOLOv8  
> **Platform:** Google Colab with T4 GPU  
---

## Section 1: Environment Setup
Install all required Python packages and verify GPU availability.

In [ ]:
# ── Install required packages ──────────────────────────────────────────
!pip install tensorflow pillow numpy pandas scikit-learn matplotlib seaborn \
            opencv-python-headless imagehash ultralytics -q

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os, json, time, hashlib, shutil, zipfile, warnings
import cv2
from PIL import Image
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, accuracy_score, precision_score, recall_score, f1_score
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print('TensorFlow version :', tf.__version__)
print('GPU devices        :', tf.config.list_physical_devices('GPU'))
print('NumPy version      :', np.__version__)

# Enable mixed precision for faster T4 training
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy().name)

# Global constants
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16
SEED       = 42
EPOCHS_MAX = 30
ARTIFACTS  = Path('/content/artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('Artifacts directory:', ARTIFACTS)


## Section 2: Google Drive Mount & Dataset Upload
Choose **Option A** (Google Drive) **or** **Option B** (direct ZIP upload).  
The ZIP file is: `Parkinson Disease detection.v1i.folder.zip`

In [ ]:
# ── Option A: Mount Google Drive ───────────────────────────────────────
# Uncomment the block below if your zip is stored in Google Drive.

# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ZIP = '/content/drive/MyDrive/Parkinson Disease detection.v1i.folder.zip'
# print('Drive mounted. ZIP path:', DRIVE_ZIP)

# ── Option B: Upload zip directly ───────────────────────────────────────
# Run this cell and use the file-picker to upload the zip.

from google.colab import files
print('Please upload: Parkinson Disease detection.v1i.folder.zip')
uploaded = files.upload()

# Locate the uploaded file
ZIP_PATH = None
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        ZIP_PATH = fname
        break

if ZIP_PATH is None:
    # Fall back to Drive path if already mounted
    if os.path.exists('/content/drive'):
        candidates = list(Path('/content/drive').rglob('*.zip'))
        if candidates:
            ZIP_PATH = str(candidates[0])

if ZIP_PATH:
    print(f'Found ZIP: {ZIP_PATH}  ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)')
else:
    raise FileNotFoundError('ZIP not found. Mount Drive or re-upload.')


## Section 3: Dataset Extraction & Validation
Extract the ZIP archive and validate that the expected split/class counts are present.

In [ ]:
# ── Extract ZIP ─────────────────────────────────────────────────────────
EXTRACT_DIR = Path('/content/dataset')
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

print(f'Extracting {ZIP_PATH} → {EXTRACT_DIR} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extraction complete.')

# ── Flatten nested single-subdirectory if present ───────────────────────
# Some zip files wrap everything in an extra folder
contents = list(EXTRACT_DIR.iterdir())
if len(contents) == 1 and contents[0].is_dir():
    inner = contents[0]
    for item in inner.iterdir():
        shutil.move(str(item), str(EXTRACT_DIR / item.name))
    inner.rmdir()
    print(f'Flattened inner directory: {inner.name}')

# ── Walk directory tree ──────────────────────────────────────────────────
SPLITS  = ['train', 'valid', 'test']
CLASSES = ['healthy', 'parkinson']
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

count_table = {}
total_images = 0

print('\nDataset directory structure:')
print('=' * 45)
for split in SPLITS:
    count_table[split] = {}
    split_path = EXTRACT_DIR / split
    if not split_path.exists():
        print(f'  WARNING: {split}/ not found!')
        continue
    for cls in CLASSES:
        cls_path = split_path / cls
        if cls_path.exists():
            imgs = [f for f in cls_path.iterdir()
                    if f.suffix.lower() in EXTENSIONS]
            count_table[split][cls] = len(imgs)
            total_images += len(imgs)
            print(f'  {split:6s} / {cls:10s}: {len(imgs):4d} images')
        else:
            count_table[split][cls] = 0
            print(f'  {split:6s} / {cls:10s}:    0 images  ⚠ MISSING')
    split_total = sum(count_table[split].values())
    print(f'  {split:6s} TOTAL              : {split_total:4d}')
    print('  ' + '-'*40)

print(f'\nGRAND TOTAL: {total_images} images')

# ── Validate expected counts ─────────────────────────────────────────────
EXPECTED = {
    'train': {'healthy': 65, 'parkinson': 68},
    'valid': {'healthy': 19, 'parkinson': 12},
    'test' : {'healthy': 18, 'parkinson': 22},
}
all_ok = True
print('\nValidation:')
for split, classes in EXPECTED.items():
    for cls, exp_count in classes.items():
        got = count_table.get(split, {}).get(cls, 0)
        status = '✓' if got == exp_count else f'✗ (expected {exp_count})'
        print(f'  {split}/{cls}: {got} {status}')
        if got != exp_count:
            all_ok = False
print('\n✓ All counts match!' if all_ok else '\n⚠ Count mismatch — please verify ZIP contents.')

DATASET_DIR = EXTRACT_DIR
TRAIN_DIR   = DATASET_DIR / 'train'
VALID_DIR   = DATASET_DIR / 'valid'
TEST_DIR    = DATASET_DIR / 'test'


## Section 4: Exploratory Data Analysis (EDA)
Full visual and statistical analysis: class distribution, sample grids, dimension analysis, pixel statistics, duplicate detection, and cross-split contamination check.

In [ ]:
# ── 4.1 Class Distribution Bar Charts ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
split_colors = {'train': ['#2196F3', '#F44336'],
                'valid': ['#4CAF50', '#FF9800'],
                'test' : ['#9C27B0', '#00BCD4']}

for ax, split in zip(axes, SPLITS):
    vals = [count_table[split].get(c, 0) for c in CLASSES]
    bars = ax.bar(CLASSES, vals, color=split_colors[split], edgecolor='black', width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(v), ha='center', va='bottom', fontweight='bold', fontsize=12)
    ax.set_title(f'{split.capitalize()} Split', fontsize=13, fontweight='bold')
    ax.set_ylabel('Image Count')
    ax.set_ylim(0, max(vals) * 1.3)
    ax.set_xlabel('Class')

plt.suptitle('Class Distribution per Split', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'eda_class_distribution.png', bbox_inches='tight')
plt.show()

# Print imbalance ratios
for split in SPLITS:
    h = count_table[split].get('healthy', 0)
    p = count_table[split].get('parkinson', 0)
    ratio = p / h if h > 0 else float('inf')
    print(f'{split:6s} → healthy={h}, parkinson={p}, ratio={ratio:.3f}')


In [ ]:
# ── 4.2 Sample Image Grid (8 healthy + 8 parkinson from train) ──────────
def load_image_samples(cls_dir, n=8):
    paths = [p for p in Path(cls_dir).iterdir()
             if p.suffix.lower() in EXTENSIONS]
    paths = sorted(paths)[:n]
    imgs  = []
    for p in paths:
        try:
            img = Image.open(p).convert('RGB').resize((128, 128))
            imgs.append((np.array(img), p.name))
        except Exception:
            pass
    return imgs

healthy_samples  = load_image_samples(TRAIN_DIR / 'healthy',  8)
parkinson_samples= load_image_samples(TRAIN_DIR / 'parkinson', 8)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
sample_data = [('Healthy', healthy_samples, '#2196F3'),
               ('Parkinson', parkinson_samples, '#F44336')]
row_offset = 0
for label, samples, color in sample_data:
    for i, (img, name) in enumerate(samples[:8]):
        r = row_offset + i // 4
        c = i % 4
        axes[r][c].imshow(img)
        axes[r][c].set_title(f'{label}\n{name[:20]}', fontsize=8,
                             color=color, fontweight='bold')
        axes[r][c].axis('off')
        for spine in axes[r][c].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2)
    row_offset += 2

plt.suptitle('Sample Images — Healthy (top 2 rows) vs Parkinson (bottom 2 rows)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'eda_sample_grid.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.3 Image Dimension Analysis ────────────────────────────────────────
dim_records = []
for split in SPLITS:
    for cls in CLASSES:
        cls_path = DATASET_DIR / split / cls
        if not cls_path.exists():
            continue
        for p in cls_path.iterdir():
            if p.suffix.lower() not in EXTENSIONS:
                continue
            try:
                with Image.open(p) as img:
                    w, h = img.size
                    dim_records.append({'split': split, 'class': cls,
                                        'width': w, 'height': h,
                                        'aspect': w/h, 'path': str(p)})
            except Exception:
                pass

df_dims = pd.DataFrame(dim_records)
print('Dimension summary:')
print(df_dims[['width','height','aspect']].describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df_dims['width'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Width Distribution'); axes[0].set_xlabel('Width (px)')

axes[1].hist(df_dims['height'], bins=30, color='darkorange', edgecolor='black')
axes[1].set_title('Height Distribution'); axes[1].set_xlabel('Height (px)')

colors = {'healthy': 'steelblue', 'parkinson': 'tomato'}
for cls in CLASSES:
    sub = df_dims[df_dims['class'] == cls]
    axes[2].scatter(sub['width'], sub['height'], alpha=0.5,
                    label=cls, c=colors[cls], s=20)
axes[2].set_title('Width vs Height Scatter')
axes[2].set_xlabel('Width (px)'); axes[2].set_ylabel('Height (px)')
axes[2].legend()

plt.suptitle('Image Dimension Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'eda_dimensions.png', bbox_inches='tight')
plt.show()

unique_dims = df_dims[['width','height']].drop_duplicates()
print(f'\nUnique (W×H) combinations: {len(unique_dims)}')
print(unique_dims.value_counts().head(10))


In [ ]:
# ── 4.4 Pixel Intensity Histograms per Class ────────────────────────────
intensity_data = {'healthy': [], 'parkinson': []}
sample_limit = 20
for cls in CLASSES:
    cls_path = TRAIN_DIR / cls
    paths = sorted([p for p in cls_path.iterdir()
                    if p.suffix.lower() in EXTENSIONS])[:sample_limit]
    for p in paths:
        try:
            arr = np.array(Image.open(p).convert('L').resize((224,224)))
            intensity_data[cls].extend(arr.flatten().tolist())
        except Exception:
            pass

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(intensity_data['healthy'],  bins=256, alpha=0.6, density=True,
        color='steelblue', label='Healthy')
ax.hist(intensity_data['parkinson'], bins=256, alpha=0.6, density=True,
        color='tomato', label='Parkinson')
ax.set_xlabel('Pixel Intensity'); ax.set_ylabel('Density')
ax.set_title('Pixel Intensity Histogram per Class (Grayscale, Train Set)')
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS / 'eda_intensity_hist.png', bbox_inches='tight')
plt.show()

# ── 4.5 RGB Channel Mean Comparison ─────────────────────────────────────
channel_means = {cls: {'R':[], 'G':[], 'B':[]} for cls in CLASSES}
for cls in CLASSES:
    cls_path = TRAIN_DIR / cls
    paths = sorted([p for p in cls_path.iterdir()
                    if p.suffix.lower() in EXTENSIONS])[:sample_limit]
    for p in paths:
        try:
            arr = np.array(Image.open(p).convert('RGB').resize((224,224)))
            channel_means[cls]['R'].append(arr[:,:,0].mean())
            channel_means[cls]['G'].append(arr[:,:,1].mean())
            channel_means[cls]['B'].append(arr[:,:,2].mean())
        except Exception:
            pass

channels = ['R', 'G', 'B']
ch_colors = ['red', 'green', 'blue']
x = np.arange(len(channels))
width = 0.3

fig, ax = plt.subplots(figsize=(8, 4))
for i, cls in enumerate(CLASSES):
    means = [np.mean(channel_means[cls][c]) for c in channels]
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, means, width, label=cls.capitalize(),
                  color=['steelblue','tomato'][i], alpha=0.8, edgecolor='black')
    for bar, m in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{m:.1f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(channels)
ax.set_title('Mean RGB Channel Values per Class (Train Set)')
ax.set_ylabel('Mean Pixel Value (0-255)')
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS / 'eda_rgb_means.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.6 Corrupt Image Detection ─────────────────────────────────────────
corrupt_images = []
total_checked  = 0
for split in SPLITS:
    for cls in CLASSES:
        cls_path = DATASET_DIR / split / cls
        if not cls_path.exists():
            continue
        for p in cls_path.iterdir():
            if p.suffix.lower() not in EXTENSIONS:
                continue
            total_checked += 1
            try:
                img = Image.open(p)
                img.verify()   # first open — verify structural integrity
            except Exception as e:
                corrupt_images.append({'path': str(p), 'error': str(e)})

print(f'Checked {total_checked} images for corruption.')
if corrupt_images:
    print(f'  ✗ {len(corrupt_images)} corrupt image(s) found:')
    for ci in corrupt_images:
        print(f'    {ci["path"]} → {ci["error"]}')
else:
    print('  ✓ No corrupt images detected.')

# ── 4.7 Exact Duplicate Detection (MD5) ─────────────────────────────────
hash_map = {}
duplicates = []
for split in SPLITS:
    for cls in CLASSES:
        cls_path = DATASET_DIR / split / cls
        if not cls_path.exists():
            continue
        for p in cls_path.iterdir():
            if p.suffix.lower() not in EXTENSIONS:
                continue
            with open(p, 'rb') as f:
                h = hashlib.md5(f.read()).hexdigest()
            key = f'{split}/{cls}/{p.name}'
            if h in hash_map:
                duplicates.append({'original': hash_map[h], 'duplicate': key, 'md5': h})
            else:
                hash_map[h] = key

print(f'\nMD5 duplicate check ({len(hash_map)} unique files):')
if duplicates:
    print(f'  ✗ {len(duplicates)} exact duplicate(s) found:')
    for d in duplicates[:10]:
        print(f'    {d["original"]}  ==  {d["duplicate"]}')
else:
    print('  ✓ No exact duplicates found.')


In [ ]:
# ── 4.8 Near-Duplicate Detection (pHash via imagehash) ──────────────────
try:
    import imagehash
    PHASH_THRESHOLD = 10  # Hamming distance ≤ 10 → near-duplicate
    phash_records = []
    for split in SPLITS:
        for cls in CLASSES:
            cls_path = DATASET_DIR / split / cls
            if not cls_path.exists():
                continue
            for p in cls_path.iterdir():
                if p.suffix.lower() not in EXTENSIONS:
                    continue
                try:
                    ph = imagehash.phash(Image.open(p))
                    phash_records.append({'key': f'{split}/{cls}/{p.name}', 'phash': ph})
                except Exception:
                    pass

    near_dups = []
    for i in range(len(phash_records)):
        for j in range(i+1, len(phash_records)):
            dist = phash_records[i]['phash'] - phash_records[j]['phash']
            if dist <= PHASH_THRESHOLD:
                near_dups.append({
                    'img1': phash_records[i]['key'],
                    'img2': phash_records[j]['key'],
                    'hamming': dist
                })

    print(f'pHash near-duplicate check (threshold ≤ {PHASH_THRESHOLD}):')
    if near_dups:
        print(f'  ✗ {len(near_dups)} near-duplicate pair(s):')
        for nd in near_dups[:10]:
            print(f'    [{nd["hamming"]}] {nd["img1"]}  ≈  {nd["img2"]}')
    else:
        print('  ✓ No near-duplicates found.')
except ImportError:
    print('imagehash not available — skipping pHash check.')

# ── 4.9 Filename Prefix Investigation ───────────────────────────────────
print('\nFilename prefix analysis (train split):')
prefix_counts = {}
for cls in CLASSES:
    cls_path = TRAIN_DIR / cls
    if not cls_path.exists():
        continue
    for p in cls_path.iterdir():
        if p.suffix.lower() not in EXTENSIONS:
            continue
        prefix = p.stem[:3].upper()  # e.g. V01, V02
        prefix_counts[prefix] = prefix_counts.get(prefix, 0) + 1

for prefix, cnt in sorted(prefix_counts.items()):
    print(f'  {prefix}: {cnt} images')

# ── 4.10 Cross-Split Duplicate Check ────────────────────────────────────
print('\nCross-split exact duplicate check:')
split_hashes = {}
for split in SPLITS:
    split_hashes[split] = set()
    for cls in CLASSES:
        cls_path = DATASET_DIR / split / cls
        if not cls_path.exists():
            continue
        for p in cls_path.iterdir():
            if p.suffix.lower() not in EXTENSIONS:
                continue
            with open(p, 'rb') as f:
                h = hashlib.md5(f.read()).hexdigest()
            split_hashes[split].add(h)

split_list = SPLITS
for i in range(len(split_list)):
    for j in range(i+1, len(split_list)):
        s1, s2 = split_list[i], split_list[j]
        overlap = split_hashes[s1] & split_hashes[s2]
        if overlap:
            print(f'  ✗ {s1} ∩ {s2}: {len(overlap)} shared image(s) — DATA LEAKAGE!')
        else:
            print(f'  ✓ {s1} ∩ {s2}: 0 shared images — clean split')


## Section 5: Preprocessing & Data Augmentation
Build TensorFlow `tf.data` pipelines. Augmentation is applied **only** during training, inside the model graph (not baked into stored images).

In [ ]:
# ── 5.1 Build TF Datasets ───────────────────────────────────────────────
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    str(TRAIN_DIR),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    label_mode='binary',
    class_names=CLASSES,
    shuffle=True,
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    str(VALID_DIR),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    label_mode='binary',
    class_names=CLASSES,
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    str(TEST_DIR),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    label_mode='binary',
    class_names=CLASSES,
    shuffle=False,
)

class_names = train_ds_raw.class_names
print('Class names (index order):', class_names)
# CLASSES[0]=healthy=0, CLASSES[1]=parkinson=1

# Performance optimization
AUTOTUNE = tf.data.AUTOTUNE

valid_ds = valid_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

print(f'Train batches : {len(train_ds_raw)}')
print(f'Valid batches : {len(valid_ds)}')
print(f'Test  batches : {len(test_ds)}')

# ── 5.2 Augmentation Layer ───────────────────────────────────────────────
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='augmentation')

# Apply augmentation only to training data
train_ds = train_ds_raw.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).cache().prefetch(AUTOTUNE)

print('Augmentation pipeline built.')

# ── 5.3 Visualize 9 Augmented Versions of One Sample ────────────────────
sample_images, sample_labels = next(iter(train_ds_raw))
sample_img = sample_images[0:1]  # shape (1, 224, 224, 3)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, ax in enumerate(axes.flatten()):
    aug_img = data_augmentation(sample_img, training=True)[0].numpy().astype('uint8')
    ax.imshow(aug_img)
    ax.set_title(f'Augmented #{i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle('9 Augmented Versions of One Training Image', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'eda_augmentation.png', bbox_inches='tight')
plt.show()


## Section 6: Custom CNN from Scratch
A lightweight 3-block CNN built entirely with Keras functional API. No pretrained weights.

In [ ]:
# ── 6.1 Build Custom CNN ────────────────────────────────────────────────
def build_cnn_model():
    inputs = tf.keras.Input(shape=(224, 224, 3), name='input_image')

    # Normalize pixel values to [0,1]
    x = tf.keras.layers.Rescaling(1./255, name='rescaling')(inputs)

    # Block 1
    x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same', name='conv1')(x)
    x = tf.keras.layers.BatchNormalization(name='bn1')(x)
    x = tf.keras.layers.MaxPooling2D(name='pool1')(x)

    # Block 2
    x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same', name='conv2')(x)
    x = tf.keras.layers.BatchNormalization(name='bn2')(x)
    x = tf.keras.layers.MaxPooling2D(name='pool2')(x)

    # Block 3
    x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same', name='conv3')(x)
    x = tf.keras.layers.BatchNormalization(name='bn3')(x)
    x = tf.keras.layers.MaxPooling2D(name='pool3')(x)

    # Classifier head
    x = tf.keras.layers.Flatten(name='flatten')(x)
    x = tf.keras.layers.Dropout(0.5, name='dropout')(x)
    x = tf.keras.layers.Dense(64, activation='relu', name='dense1')(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)

    model = tf.keras.Model(inputs, outputs, name='custom_cnn')
    return model

cnn_model = build_cnn_model()
cnn_model.summary()

# ── 6.2 Compile ─────────────────────────────────────────────────────────
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# ── 6.3 Callbacks ───────────────────────────────────────────────────────
cnn_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS / 'cnn_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
]

# ── 6.4 Train ───────────────────────────────────────────────────────────
print('Training Custom CNN...')
cnn_start = time.time()
cnn_history = cnn_model.fit(
    train_ds,
    epochs=EPOCHS_MAX,
    validation_data=valid_ds,
    callbacks=cnn_callbacks,
    verbose=1
)
cnn_train_time = time.time() - cnn_start
print(f'\nCNN training time: {cnn_train_time/60:.1f} minutes')
print(f'Best val_loss   : {min(cnn_history.history["val_loss"]):.4f}')
print(f'Best val_acc    : {max(cnn_history.history["val_accuracy"]):.4f}')


## Section 7: VGG16 Transfer Learning
**Phase 1:** Freeze the entire VGG16 base and train only the new head (feature extraction).  
**Phase 2:** Unfreeze the last 20 layers and fine-tune with a very low learning rate.

In [ ]:
# ── 7.1 Build VGG16 Model — Feature Extraction Phase ───────────────────
def build_vgg16_model(trainable_base=False):
    base = tf.keras.applications.VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base.trainable = trainable_base

    inputs = tf.keras.Input(shape=(224, 224, 3), name='input_image')
    x = tf.keras.applications.vgg16.preprocess_input(inputs)  # built-in normalization
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = tf.keras.layers.Dense(256, activation='relu', name='dense1')(x)
    x = tf.keras.layers.Dropout(0.5, name='dropout')(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)

    model = tf.keras.Model(inputs, outputs, name='vgg16_transfer')
    return model, base

vgg16_model, vgg16_base = build_vgg16_model(trainable_base=False)

vgg16_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)
print('VGG16 Feature Extraction model:')
print(f'  Trainable params: {vgg16_model.count_params():,}')

vgg16_callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS / 'vgg16_phase1_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
]

print('\nPhase 1 — Feature Extraction...')
vgg16_start = time.time()
vgg16_history_p1 = vgg16_model.fit(
    train_ds,
    epochs=15,
    validation_data=valid_ds,
    callbacks=vgg16_callbacks_p1,
    verbose=1
)
print(f'Phase 1 done. Best val_loss: {min(vgg16_history_p1.history["val_loss"]):.4f}')

# ── 7.2 Phase 2 — Fine-Tune Last 20 Layers ──────────────────────────────
vgg16_base.trainable = True
# Freeze all but the last 20 layers
for layer in vgg16_base.layers[:-20]:
    layer.trainable = False

trainable_now = sum(1 for l in vgg16_base.layers if l.trainable)
print(f'\nPhase 2: {trainable_now} VGG16 layers unfrozen for fine-tuning')

vgg16_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

vgg16_callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4,
        min_lr=1e-8, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS / 'vgg16_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
]

print('Phase 2 — Fine-tuning...')
vgg16_history_p2 = vgg16_model.fit(
    train_ds,
    epochs=20,
    validation_data=valid_ds,
    callbacks=vgg16_callbacks_p2,
    verbose=1
)
vgg16_train_time = time.time() - vgg16_start
print(f'VGG16 total training time: {vgg16_train_time/60:.1f} minutes')
print(f'Best val_loss (fine-tune): {min(vgg16_history_p2.history["val_loss"]):.4f}')


## Section 8: ResNet50 Transfer Learning
Same two-phase approach as VGG16. BatchNormalization layers are explicitly frozen during fine-tuning to preserve ImageNet statistics.

In [ ]:
# ── 8.1 Build ResNet50 Model — Feature Extraction Phase ─────────────────
def build_resnet50_model(trainable_base=False):
    base = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base.trainable = trainable_base

    inputs = tf.keras.Input(shape=(224, 224, 3), name='input_image')
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = tf.keras.layers.Dense(256, activation='relu', name='dense1')(x)
    x = tf.keras.layers.Dropout(0.5, name='dropout')(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)

    model = tf.keras.Model(inputs, outputs, name='resnet50_transfer')
    return model, base

resnet50_model, resnet50_base = build_resnet50_model(trainable_base=False)

resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)
print('ResNet50 Feature Extraction model:')
print(f'  Trainable params: {resnet50_model.count_params():,}')

resnet50_callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS / 'resnet50_phase1_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
]

print('\nPhase 1 — Feature Extraction...')
resnet50_start = time.time()
resnet50_history_p1 = resnet50_model.fit(
    train_ds,
    epochs=15,
    validation_data=valid_ds,
    callbacks=resnet50_callbacks_p1,
    verbose=1
)
print(f'Phase 1 done. Best val_loss: {min(resnet50_history_p1.history["val_loss"]):.4f}')

# ── 8.2 Phase 2 — Fine-Tune Last 20 Layers (freeze BatchNorm) ───────────
resnet50_base.trainable = True
for layer in resnet50_base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False   # Keep BN stats frozen

# Only fine-tune last 20 non-BN layers
non_bn_layers = [l for l in resnet50_base.layers
                 if not isinstance(l, tf.keras.layers.BatchNormalization)]
for layer in non_bn_layers[:-20]:
    layer.trainable = False

trainable_now = sum(1 for l in resnet50_base.layers if l.trainable)
print(f'\nPhase 2: {trainable_now} ResNet50 layers unfrozen (BN layers kept frozen)')

resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

resnet50_callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4,
        min_lr=1e-8, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS / 'resnet50_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
]

print('Phase 2 — Fine-tuning (BN frozen)...')
resnet50_history_p2 = resnet50_model.fit(
    train_ds,
    epochs=20,
    validation_data=valid_ds,
    callbacks=resnet50_callbacks_p2,
    verbose=1
)
resnet50_train_time = time.time() - resnet50_start
print(f'ResNet50 total training time: {resnet50_train_time/60:.1f} minutes')
print(f'Best val_loss (fine-tune)   : {min(resnet50_history_p2.history["val_loss"]):.4f}')


## Section 9: Model Evaluation on Test Set
Evaluate all three models with accuracy, precision, recall, F1, specificity, ROC-AUC, confusion matrices, and ROC curves.

In [ ]:
# ── 9.0 Helper: collect test predictions ────────────────────────────────
def get_test_predictions(model, test_dataset):
    """Return (y_true, y_pred_prob, y_pred_binary) numpy arrays."""
    y_true_all, y_prob_all = [], []
    for images, labels in test_dataset:
        probs = model.predict(images, verbose=0)
        y_true_all.extend(labels.numpy().flatten())
        y_prob_all.extend(probs.flatten())
    y_true  = np.array(y_true_all)
    y_prob  = np.array(y_prob_all)
    y_pred  = (y_prob >= 0.5).astype(int)
    return y_true, y_prob, y_pred

def compute_all_metrics(y_true, y_prob, y_pred, model_name):
    """Compute and print all classification metrics."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_prob)
    cm   = confusion_matrix(y_true, y_pred)
    # Specificity = TN / (TN + FP)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    print(f'\n{'='*50}')
    print(f'Model: {model_name}')
    print(f'  Accuracy   : {acc:.4f}')
    print(f'  Precision  : {prec:.4f}')
    print(f'  Recall     : {rec:.4f}')
    print(f'  F1-Score   : {f1:.4f}')
    print(f'  Specificity: {spec:.4f}')
    print(f'  ROC-AUC    : {auc:.4f}')
    print(f'  Confusion Matrix:\n{cm}')
    print(classification_report(y_true, y_pred,
                                target_names=CLASSES, zero_division=0))
    return {'model': model_name, 'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1, 'specificity': spec, 'roc_auc': auc,
            'cm': cm, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

# ── 9.1 Evaluate all models ──────────────────────────────────────────────
models_to_eval = [
    ('Custom CNN',  cnn_model),
    ('VGG16',       vgg16_model),
    ('ResNet50',    resnet50_model),
]

all_preds   = {}  # {name: (y_true, y_prob, y_pred)}
all_metrics = []  # list of metric dicts

for model_name, model in models_to_eval:
    y_true, y_prob, y_pred = get_test_predictions(model, test_ds)
    all_preds[model_name] = (y_true, y_prob, y_pred)
    metrics = compute_all_metrics(y_true, y_prob, y_pred, model_name)
    all_metrics.append(metrics)

print('\n✓ All models evaluated on test set.')


In [ ]:
# ── 9.2 Confusion Matrices ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap_list = ['Blues', 'Greens', 'Purples']

for ax, metrics, cmap in zip(axes, all_metrics, cmap_list):
    cm = metrics['cm']
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES,
                linewidths=0.5, annot_kws={'size': 14})
    ax.set_title(f"{metrics['model']}\n"
                 f"Acc={metrics['accuracy']:.3f} | F1={metrics['f1']:.3f}",
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'eval_confusion_matrices.png', bbox_inches='tight')
plt.show()

# ── 9.3 ROC Curves ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colors  = ['steelblue', 'darkorange', 'green']
linestyles = ['-', '--', '-.']

for (model_name, (y_true, y_prob, _)), color, ls in zip(all_preds.items(), colors, linestyles):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, linestyle=ls,
            label=f'{model_name} (AUC = {auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Test Set Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'eval_roc_curves.png', bbox_inches='tight')
plt.show()


## Section 10: Model Comparison Bar Charts
Side-by-side comparison of all models across all evaluation metrics.

In [ ]:
# ── 10 Model Comparison Bar Charts ─────────────────────────────────────
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'specificity', 'roc_auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity', 'ROC-AUC']
model_names   = [m['model'] for m in all_metrics]
bar_colors    = ['#2196F3', '#FF9800', '#4CAF50']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, metric, label in zip(axes, metric_names, metric_labels):
    values = [m[metric] for m in all_metrics]
    bars = ax.bar(model_names, values, color=bar_colors, edgecolor='black', width=0.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=15)
    ax.axhline(0.9, color='red', linestyle='--', alpha=0.4, lw=1, label='0.9 target')
    ax.legend(fontsize=7)

plt.suptitle('Model Performance Comparison (Test Set)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'eval_model_comparison.png', bbox_inches='tight')
plt.show()

# Summary table
df_metrics = pd.DataFrame(all_metrics)[['model'] + metric_names]
df_metrics.columns = ['Model'] + metric_labels
df_metrics = df_metrics.set_index('Model')
print('\nModel Comparison Table:')
print(df_metrics.round(4).to_string())


## Section 11: Training History Plots
Loss and accuracy learning curves for each model. VGG16 and ResNet50 show both training phases.

In [ ]:
# ── 11 Training History Plots ───────────────────────────────────────────
def plot_history(histories, model_name, title_prefix=''):
    """
    histories: list of keras History objects (1 for CNN, 2 for VGG16/ResNet50)
    """
    # Concatenate histories
    combined = {}
    for h in histories:
        for key, vals in h.history.items():
            combined.setdefault(key, []).extend(vals)

    epochs = range(1, len(combined['loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Phase boundary
    phase_boundary = None
    if len(histories) > 1:
        phase_boundary = len(histories[0].history['loss'])

    # Loss
    axes[0].plot(epochs, combined['loss'],     'b-o', ms=3, label='Train Loss')
    axes[0].plot(epochs, combined['val_loss'], 'r-o', ms=3, label='Val Loss')
    if phase_boundary:
        axes[0].axvline(phase_boundary, color='gray', linestyle='--', lw=1.5,
                        label=f'Phase boundary (ep {phase_boundary})')
    axes[0].set_title(f'{title_prefix} Loss Curve', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(epochs, combined['accuracy'],     'b-o', ms=3, label='Train Acc')
    axes[1].plot(epochs, combined['val_accuracy'], 'r-o', ms=3, label='Val Acc')
    if phase_boundary:
        axes[1].axvline(phase_boundary, color='gray', linestyle='--', lw=1.5,
                        label=f'Phase boundary (ep {phase_boundary})')
    axes[1].set_title(f'{title_prefix} Accuracy Curve', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'{model_name} Training History', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fname = model_name.lower().replace(' ', '_').replace('/', '_')
    plt.savefig(ARTIFACTS / f'history_{fname}.png', bbox_inches='tight')
    plt.show()

plot_history([cnn_history],                            'Custom CNN')
plot_history([vgg16_history_p1, vgg16_history_p2],     'VGG16',    'Phase 1 & 2')
plot_history([resnet50_history_p1, resnet50_history_p2],'ResNet50', 'Phase 1 & 2')


## Section 12: Error Analysis
Visualize the misclassified test images for the best-performing model, with true labels, predicted labels, and confidence scores.

In [ ]:
# ── 12 Error Analysis ────────────────────────────────────────────────────
# Use VGG16 for error analysis (typically the strongest transfer model)
# Collect test image paths + labels in order
test_image_paths = []
test_labels_ordered = []
for cls_idx, cls in enumerate(CLASSES):
    cls_path = TEST_DIR / cls
    if cls_path.exists():
        for p in sorted(cls_path.iterdir()):
            if p.suffix.lower() in EXTENSIONS:
                test_image_paths.append(p)
                test_labels_ordered.append(cls_idx)

# Collect predictions from VGG16 on test set (unbatched)
vgg16_probs_all = []
for p in test_image_paths:
    img = tf.keras.utils.load_img(str(p), target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    arr = np.expand_dims(arr, 0)
    prob = vgg16_model.predict(arr, verbose=0)[0][0]
    vgg16_probs_all.append(float(prob))

y_true_arr = np.array(test_labels_ordered)
y_prob_arr = np.array(vgg16_probs_all)
y_pred_arr = (y_prob_arr >= 0.5).astype(int)

# Find misclassified indices
misclassified_idx = np.where(y_true_arr != y_pred_arr)[0]
print(f'VGG16 misclassified {len(misclassified_idx)} / {len(y_true_arr)} test images')

if len(misclassified_idx) == 0:
    print('Perfect classification on test set — no errors to show!')
else:
    n_show = min(len(misclassified_idx), 12)
    ncols  = 4
    nrows  = (n_show + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
    if nrows == 1:
        axes = [axes]
    axes_flat = [ax for row in axes for ax in (row if hasattr(row, '__iter__') else [row])]

    for i, idx in enumerate(misclassified_idx[:n_show]):
        ax = axes_flat[i]
        img = Image.open(test_image_paths[idx]).convert('RGB').resize((224, 224))
        ax.imshow(img)
        true_label = CLASSES[y_true_arr[idx]]
        pred_label = CLASSES[y_pred_arr[idx]]
        score      = y_prob_arr[idx]
        ax.set_title(
            f'True: {true_label}\nPred: {pred_label}\nScore: {score:.3f}',
            fontsize=8,
            color='red' if true_label != pred_label else 'green'
        )
        ax.axis('off')

    # Hide unused axes
    for j in range(n_show, len(axes_flat)):
        axes_flat[j].axis('off')

    plt.suptitle('Misclassified Test Images (VGG16)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(ARTIFACTS / 'eval_error_analysis.png', bbox_inches='tight')
    plt.show()


## Section 13: Grad-CAM Visualization
Gradient-weighted Class Activation Mapping (Grad-CAM) highlights the regions the model focuses on when making a prediction.

In [ ]:
# ── 13 Grad-CAM Implementation ──────────────────────────────────────────
import tensorflow as tf
import numpy as np

def get_last_conv_layer_name(model):
    """Find the last Conv2D layer name in the model (or its base sub-model)."""
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
        # Handle nested sub-models (e.g. VGG16 / ResNet50 base)
        if hasattr(layer, 'layers'):
            for sub_layer in reversed(layer.layers):
                if isinstance(sub_layer, tf.keras.layers.Conv2D):
                    return layer.name  # return the sub-model's name
    return None

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    img_array : numpy array shape (1, H, W, 3)  — raw uint8 or float
    Returns   : heatmap numpy array shape (h, w) in [0, 1]
    """
    # Build a model that outputs (last_conv_output, predictions)
    grad_model = tf.keras.models.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array, tf.float32)
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, 0]  # binary

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE)
    heatmap_resized = cv2.resize(heatmap, IMG_SIZE)
    heatmap_uint8   = np.uint8(255 * heatmap_resized)
    heatmap_color   = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color   = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    superimposed   = cv2.addWeighted(img, 1 - alpha, heatmap_color, alpha, 0)
    return img, heatmap_resized, superimposed

# ── Apply Grad-CAM to VGG16 (or best model) ─────────────────────────────
# For VGG16 transfer model, the conv layers are inside the vgg16_base sub-model
# We'll use the named base layer
GRADCAM_MODEL    = vgg16_model
GRADCAM_LAYER    = 'vgg16'           # sub-model name in the functional model

# Try inner conv layer for standalone CNN
cnn_last_conv = get_last_conv_layer_name(cnn_model)
print(f'CNN last conv layer: {cnn_last_conv}')

# Pick a few test images to visualize
sample_paths  = []
sample_labels_gc = []
for cls in CLASSES:
    cls_path = TEST_DIR / cls
    if cls_path.exists():
        ps = sorted([p for p in cls_path.iterdir()
                     if p.suffix.lower() in EXTENSIONS])[:2]
        sample_paths.extend(ps)
        sample_labels_gc.extend([cls] * len(ps))

# Grad-CAM on Custom CNN (last conv layer is accessible)
if cnn_last_conv:
    n_gc = len(sample_paths)
    fig, axes = plt.subplots(n_gc, 3, figsize=(12, 4 * n_gc))
    if n_gc == 1:
        axes = [axes]

    for i, (img_path, true_cls) in enumerate(zip(sample_paths, sample_labels_gc)):
        img_arr  = tf.keras.utils.img_to_array(
                       tf.keras.utils.load_img(str(img_path), target_size=IMG_SIZE))
        img_arr  = np.expand_dims(img_arr, 0)

        heatmap  = make_gradcam_heatmap(img_arr, cnn_model, cnn_last_conv)
        orig_img, hm, superimposed = overlay_gradcam(img_path, heatmap)

        pred_prob = cnn_model.predict(img_arr, verbose=0)[0][0]
        pred_cls  = CLASSES[int(pred_prob >= 0.5)]

        axes[i][0].imshow(orig_img)
        axes[i][0].set_title(f'Original\nTrue: {true_cls}', fontsize=9)
        axes[i][0].axis('off')

        axes[i][1].imshow(hm, cmap='jet')
        axes[i][1].set_title(f'Grad-CAM Heatmap\nPred: {pred_cls} ({pred_prob:.3f})', fontsize=9)
        axes[i][1].axis('off')

        axes[i][2].imshow(superimposed)
        axes[i][2].set_title('Overlay', fontsize=9)
        axes[i][2].axis('off')

    plt.suptitle('Grad-CAM — Custom CNN (Last Conv Layer)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(ARTIFACTS / 'gradcam_cnn.png', bbox_inches='tight')
    plt.show()
else:
    print('Could not find a Conv2D layer in CNN model for Grad-CAM.')


## Section 14: Best Model Selection
**Primary criterion:** Parkinson-class F1-Score (maximises recall for the positive class — missing a Parkinson case is more costly than a false alarm).
**Secondary criterion:** Parkinson-class Recall (sensitivity).

The model with the highest F1 (and, as a tiebreaker, the highest recall) is selected as the production model.

In [ ]:
# ── 14 Best Model Selection ─────────────────────────────────────────────
# Re-compute per-class F1 and recall for parkinson class (class index 1)
selection_records = []
for model_name, (y_true, y_prob, y_pred) in all_preds.items():
    report = classification_report(
        y_true, y_pred,
        target_names=CLASSES,
        output_dict=True,
        zero_division=0
    )
    pk_f1     = report['parkinson']['f1-score']
    pk_recall = report['parkinson']['recall']
    overall_f1 = f1_score(y_true, y_pred, zero_division=0)
    overall_auc = roc_auc_score(y_true, y_prob)
    selection_records.append({
        'Model'      : model_name,
        'PK_F1'      : pk_f1,
        'PK_Recall'  : pk_recall,
        'Overall_F1' : overall_f1,
        'ROC_AUC'    : overall_auc,
    })

df_sel = pd.DataFrame(selection_records).set_index('Model')
print('Model Selection Table (sorted by Parkinson F1 descending):')
print(df_sel.sort_values('PK_F1', ascending=False).round(4).to_string())

# Select best
best_row   = df_sel.sort_values(['PK_F1','PK_Recall'], ascending=False).iloc[0]
best_name  = best_row.name
best_model = dict(zip([m_name for m_name, _ in models_to_eval],
                       [m for _, m in models_to_eval]))[best_name]

print(f'\n🏆 Best model selected: {best_name}')
print(f'   Parkinson F1-Score : {best_row["PK_F1"]:.4f}')
print(f'   Parkinson Recall   : {best_row["PK_Recall"]:.4f}')
print(f'   ROC-AUC            : {best_row["ROC_AUC"]:.4f}')

BEST_MODEL_NAME = best_name
best_model.save(str(ARTIFACTS / 'best_model.keras'))
print(f'\n✓ Best model saved to {ARTIFACTS}/best_model.keras')


## Section 15: Save All Artifacts
Save model weights, evaluation metrics, training histories, and class mappings as JSON. Optionally copy everything to Google Drive.

In [ ]:
# ── 15.1 Save all Keras models ──────────────────────────────────────────
cnn_model.save(str(ARTIFACTS / 'cnn_model.keras'))
vgg16_model.save(str(ARTIFACTS / 'vgg16_model.keras'))
resnet50_model.save(str(ARTIFACTS / 'resnet50_model.keras'))
print('All Keras models saved.')

# ── 15.2 Save model metadata JSON ───────────────────────────────────────
model_metadata = {
    'project'      : 'NEUROSCAN AI — Parkinson Disease Detection',
    'created_at'   : time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'best_model'   : BEST_MODEL_NAME,
    'class_mapping': {'0': 'healthy', '1': 'parkinson'},
    'image_size'   : list(IMG_SIZE),
    'batch_size'   : BATCH_SIZE,
    'models': {
        'custom_cnn': {
            'file'          : 'cnn_model.keras',
            'train_time_s'  : round(cnn_train_time, 2),
            'epochs_trained': len(cnn_history.history['loss']),
            'architecture'  : 'Custom 3-block CNN',
            'preprocessing' : 'Rescaling(1/255)',
        },
        'vgg16': {
            'file'          : 'vgg16_model.keras',
            'train_time_s'  : round(vgg16_train_time, 2),
            'epochs_p1'     : len(vgg16_history_p1.history['loss']),
            'epochs_p2'     : len(vgg16_history_p2.history['loss']),
            'architecture'  : 'VGG16 pretrained (ImageNet)',
            'preprocessing' : 'vgg16.preprocess_input',
            'fine_tuned_layers': 'last 20',
        },
        'resnet50': {
            'file'          : 'resnet50_model.keras',
            'train_time_s'  : round(resnet50_train_time, 2),
            'epochs_p1'     : len(resnet50_history_p1.history['loss']),
            'epochs_p2'     : len(resnet50_history_p2.history['loss']),
            'architecture'  : 'ResNet50 pretrained (ImageNet)',
            'preprocessing' : 'resnet50.preprocess_input',
            'fine_tuned_layers': 'last 20 non-BN layers',
        },
    }
}

with open(ARTIFACTS / 'model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)
print('model_metadata.json saved.')

# ── 15.3 Save evaluation results JSON ───────────────────────────────────
evaluation_results = {}
for m in all_metrics:
    name = m['model']
    evaluation_results[name] = {
        'accuracy'   : round(m['accuracy'], 6),
        'precision'  : round(m['precision'], 6),
        'recall'     : round(m['recall'], 6),
        'f1'         : round(m['f1'], 6),
        'specificity': round(m['specificity'], 6),
        'roc_auc'    : round(m['roc_auc'], 6),
        'confusion_matrix': {
            'TN': m['tn'], 'FP': m['fp'],
            'FN': m['fn'], 'TP': m['tp']
        }
    }

with open(ARTIFACTS / 'evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)
print('evaluation_results.json saved.')

# ── 15.4 Save training history JSON ─────────────────────────────────────
def convert_history(h):
    return {k: [float(x) for x in v] for k, v in h.history.items()}

training_history = {
    'custom_cnn': convert_history(cnn_history),
    'vgg16_phase1': convert_history(vgg16_history_p1),
    'vgg16_phase2': convert_history(vgg16_history_p2),
    'resnet50_phase1': convert_history(resnet50_history_p1),
    'resnet50_phase2': convert_history(resnet50_history_p2),
}

with open(ARTIFACTS / 'training_history.json', 'w') as f:
    json.dump(training_history, f, indent=2)
print('training_history.json saved.')

# ── 15.5 Save class mapping ──────────────────────────────────────────────
class_mapping = {'0': 'healthy', '1': 'parkinson'}
with open(ARTIFACTS / 'class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)
print('class_mapping.json saved.')

# ── 15.6 List all artifacts ──────────────────────────────────────────────
print('\nAll artifacts in', ARTIFACTS, ':')
for p in sorted(ARTIFACTS.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:<40} {size_mb:.2f} MB')

# ── 15.7 Copy to Google Drive (if mounted) ───────────────────────────────
DRIVE_DST = '/content/drive/MyDrive/NEUROSCAN_AI_artifacts'
if os.path.exists('/content/drive/MyDrive'):
    if os.path.exists(DRIVE_DST):
        shutil.rmtree(DRIVE_DST)
    shutil.copytree(str(ARTIFACTS), DRIVE_DST)
    print(f'\n✓ Artifacts copied to Google Drive: {DRIVE_DST}')
else:
    print('\nGoogle Drive not mounted — skipping Drive upload.')
    print('To save to Drive, mount it in Section 2 (Option A) and re-run this cell.')


## Section 16: Optional YOLOv8 Classification Baseline
Train YOLOv8-nano classifier as a quick baseline. **This model is NOT a replacement** for CNN/VGG16/ResNet50; it is an additional comparison point only.

> **Note:** YOLOv8 classification expects `train/`, `val/`, `test/` directories with class subdirectories — which our dataset already provides.

In [ ]:
# ── 16 YOLOv8 Classification Baseline ──────────────────────────────────
# This section is OPTIONAL. Remove or skip if ultralytics is not available.

try:
    from ultralytics import YOLO
    import yaml

    # YOLOv8 uses 'val' directory (not 'valid')
    # Create a symlink if our dir is named 'valid'
    yolo_data_dir = Path('/content/dataset')
    val_link      = yolo_data_dir / 'val'
    if not val_link.exists() and (yolo_data_dir / 'valid').exists():
        val_link.symlink_to(yolo_data_dir / 'valid')
        print('Created val → valid symlink for YOLOv8')

    print('Loading YOLOv8-nano classification model...')
    model_yolo = YOLO('yolov8n-cls.pt')

    print('\nTraining YOLOv8-nano classifier (15 epochs)...')
    yolo_start = time.time()
    yolo_results = model_yolo.train(
        data   = str(yolo_data_dir),
        epochs = 15,
        imgsz  = 224,
        batch  = 16,
        project= str(ARTIFACTS / 'yolo_runs'),
        name   = 'parkinson_cls',
        exist_ok=True,
        verbose= False,
    )
    yolo_train_time = time.time() - yolo_start
    print(f'YOLOv8 training time: {yolo_train_time/60:.1f} minutes')

    # ── Evaluate on test set ─────────────────────────────────────────────
    print('\nEvaluating YOLOv8 on test set...')
    yolo_y_true, yolo_y_prob = [], []
    for cls_idx, cls in enumerate(CLASSES):
        cls_path = TEST_DIR / cls
        if not cls_path.exists():
            continue
        for p in sorted(cls_path.iterdir()):
            if p.suffix.lower() not in EXTENSIONS:
                continue
            res = model_yolo.predict(str(p), verbose=False)[0]
            probs = res.probs.data.cpu().numpy()
            # parkinson is class index 1
            prob_parkinson = float(probs[1]) if len(probs) > 1 else float(probs[0])
            yolo_y_true.append(cls_idx)
            yolo_y_prob.append(prob_parkinson)

    yolo_y_true = np.array(yolo_y_true)
    yolo_y_prob = np.array(yolo_y_prob)
    yolo_y_pred = (yolo_y_prob >= 0.5).astype(int)

    yolo_acc  = accuracy_score(yolo_y_true, yolo_y_pred)
    yolo_f1   = f1_score(yolo_y_true, yolo_y_pred, zero_division=0)
    yolo_auc  = roc_auc_score(yolo_y_true, yolo_y_prob)
    print(f'\nYOLOv8-nano Baseline Results (test set):')
    print(f'  Accuracy : {yolo_acc:.4f}')
    print(f'  F1-Score : {yolo_f1:.4f}')
    print(f'  ROC-AUC  : {yolo_auc:.4f}')
    print(classification_report(yolo_y_true, yolo_y_pred,
                                target_names=CLASSES, zero_division=0))

    # Save YOLOv8 metrics
    yolo_metrics = {'accuracy': float(yolo_acc), 'f1': float(yolo_f1), 'roc_auc': float(yolo_auc)}
    with open(ARTIFACTS / 'yolo_evaluation.json', 'w') as f:
        json.dump(yolo_metrics, f, indent=2)
    print('yolo_evaluation.json saved.')

except ImportError:
    print('ultralytics not installed — skipping YOLOv8 baseline.')
    print('Install with: !pip install ultralytics')
except Exception as e:
    print(f'YOLOv8 training failed: {e}')
    print('This is a non-critical optional section. Continuing...')


## 🎉 Training Complete!

| Artifact | Location |
|---|---|
| CNN model | `/content/artifacts/cnn_model.keras` |
| VGG16 model | `/content/artifacts/vgg16_model.keras` |
| ResNet50 model | `/content/artifacts/resnet50_model.keras` |
| Best model | `/content/artifacts/best_model.keras` |
| Evaluation JSON | `/content/artifacts/evaluation_results.json` |
| Training history | `/content/artifacts/training_history.json` |
| Class mapping | `/content/artifacts/class_mapping.json` |
| Model metadata | `/content/artifacts/model_metadata.json` |
| All plots | `/content/artifacts/*.png` |

> **Next step:** Use `best_model.keras` and `class_mapping.json` in the NEUROSCAN AI FastAPI inference server.